# EOM calibration parameters over time

**Run All** with the `niDaqENV` Python kernel. This notebook reads saved analyses only: it does not refit data, connect to instruments or change calibration settings. Edit the settings cell if needed.

The three panels show **Alice Vπ**, **Bob Vπ**, and the **combined static phase offset**. Each current global-fit `analysis.json` saves `fit.vpi_alice_v`, `fit.vpi_bob_v`, and `phase_rad` in the named `fit.parameters` array.

The model is `CH1 = C1 + A1 sin²(kA VA + kB VB + φ)`. The default phase plot shows the **raw stored φ in radians**. To display the full retardance offset, select **2φ** in radians or degrees below. Only one combined offset is identifiable in this model; there are not separate fitted static offsets for Alice and Bob.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import importlib
import matplotlib.pyplot as plt
from IPython.display import display

# Find the repository when launched from its root or the EOM_scripts folder.
candidates = [Path.cwd(), *Path.cwd().parents,
              Path(r'C:\Users\nanometa\Documents\QKD_Code')]
REPO_ROOT = next((p for p in candidates if (p / 'pulsed_GUI' / 'calibration_history.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open this notebook from QKD_Code, or add the repository path to candidates above.')
sys.path.insert(0, str(REPO_ROOT / 'pulsed_GUI'))
import calibration_history
importlib.reload(calibration_history)
print('Repository:', REPO_ROOT)


## Settings

Defaults show **every completed global fit at its analysis time**, including repeated analyses of the same sweep. For measurement-to-measurement drift, set `TIME_AXIS = 'sweep'` and `LATEST_PER_SWEEP = True`. Sweep time means the UTC timestamp in the acquisition folder name, not the exact first photon time.

Fit diagnostics remain visible by default. Fine-tuning reports are excluded because their Vπ values are inherited, not refitted; old measured-grid analyses do not contain these fitted parameters. The excluded-file table below explains every omission.


In [ ]:
DATA_FOLDER = REPO_ROOT / 'data' / 'eoms'  # Searches subfolders automatically.
TIME_AXIS = 'analysis'                     # 'analysis' or 'sweep'
LATEST_PER_SWEEP = False                    # True: latest analysis of each source sweep only
ONLY_VALIDATED = False                      # True: keep only identified fits passing the saved model check
TIME_ZONE = 'America/New_York'              # Or 'UTC'
PHASE_DISPLAY = 'stored_phi_rad'            # 'stored_phi_rad', 'retardance_rad', 'retardance_deg'
SAVE_OUTPUTS = True
OUTPUT_FOLDER = DATA_FOLDER / 'parameter_history'
FIGURE_DPI = 300


In [ ]:
all_results, excluded = calibration_history.load_history(DATA_FOLDER)
print(f'Found {len(all_results)} completed global fits; excluded {len(excluded)} other/incomplete/invalid analyses.')
if not excluded.empty:
    display(excluded)

results, missing_times = calibration_history.select_history(
    all_results, time_axis=TIME_AXIS, latest_per_sweep=LATEST_PER_SWEEP,
    only_validated=ONLY_VALIDATED)
print(f'Plotting {len(results)} analyses from {results.sweep_key.nunique()} distinct sweeps.')
if missing_times:
    print(f'{missing_times} fits omitted from this plot because the selected timestamp is unavailable.')
print(f'{int(results.flagged.sum())} plotted fits have diagnostics or did not pass the saved Poisson model check.')
if results.sweep_key.duplicated().any():
    print('Repeated fits of the same sweep are not independent measurements; set LATEST_PER_SWEEP=True to remove them.')
if results.k_bob_rad_per_v.apply(lambda x: x > 0).nunique() > 1:
    print('Bob phase-slope sign changes between fits. Inspect phase branches before interpreting apparent drift.')

summary = results[['plot_time_utc', 'sweep_id', 'vpi_alice_v', 'vpi_bob_v', 'phase_rad',
                   'static_retardance_deg', 'fit_identified', 'poisson_model_check_passed', 'diagnostics']].copy()
summary['time_local'] = summary.pop('plot_time_utc').dt.tz_convert(TIME_ZONE)
display(summary[['time_local', *[c for c in summary.columns if c != 'time_local']]])


## Three parameter traces

Points are the saved raw fit values; no averaging, smoothing, phase unwrapping or new curve fitting is applied. φ is stored modulo π (2φ modulo 2π). Lines break at large phase wraps or a Bob-slope sign change; do not interpret these branch changes as physical drift. Connecting lines are guides only.

Red crosses mark existing fit diagnostics or failed model checks. They are retained so you can inspect the history, not presented as validated optical calibration. Changing fit bounds, model versions, source conditions or optical alignment can also change these fitted parameters.


In [ ]:
fig = calibration_history.plot_history(
    results, time_axis=TIME_AXIS, timezone=TIME_ZONE,
    phase_display=PHASE_DISPLAY, dpi=FIGURE_DPI)
plt.show()


## Save the figure and tables

Each execution saves to a new UTC-stamped folder under `data/eoms/parameter_history/`. Original measurement files are unchanged. `all_global_fits.csv` includes every loaded analysis; `plotted_fits.csv` contains the selected rows, source paths, raw φ and both 2φ unit conventions. Set `SAVE_OUTPUTS = False` above for display only.


In [ ]:
if SAVE_OUTPUTS:
    output = OUTPUT_FOLDER / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    output.mkdir(parents=True, exist_ok=False)
    fig.savefig(output / 'calibration_parameter_history.png', dpi=FIGURE_DPI)
    fig.savefig(output / 'calibration_parameter_history.pdf')
    all_results.to_csv(output / 'all_global_fits.csv', index=False)
    results.to_csv(output / 'plotted_fits.csv', index=False)
    excluded.to_csv(output / 'excluded_analyses.csv', index=False)
    settings = dict(data_folder=str(DATA_FOLDER.resolve()), time_axis=TIME_AXIS,
                    latest_per_sweep=LATEST_PER_SWEEP, only_validated=ONLY_VALIDATED,
                    timezone=TIME_ZONE, phase_display=PHASE_DISPLAY, figure_dpi=FIGURE_DPI,
                    phase_convention='Stored phi is the sin-squared argument offset; full retardance offset is 2*phi.',
                    plotted_analysis_files=results.analysis_file.tolist())
    (output / 'history_settings.json').write_text(json.dumps(settings, indent=2), encoding='utf-8')
    print('Saved plots and tables to:', output)
else:
    print('Display only; no output files saved.')
